# Superstore Retail Sales Performance & Profit Optimization

**Objective:** Clean, explore, analyze, and translate Superstore retail transactions into business insights for the companion MySQL analysis and Power BI dashboard.

This notebook focuses on reproducible Python analysis: data quality, KPI development, sales/profitability trends, product and market performance, discount behavior, and optimization opportunities.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load Dataset

In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd().parent
raw_path = BASE_DIR / "data" / "raw" / "superstore.csv"
clean_path = BASE_DIR / "data" / "processed" / "superstore_cleaned.csv"

df = pd.read_csv(raw_path)
print("Shape:", df.shape)
df.head()

## 3. Dataset Overview

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
df.info()

In [ ]:
df.describe(include="all").T

## 4. Data Quality Checks

In [ ]:
quality = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_values", ascending=False)
quality

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate Row IDs:", df["Row ID"].duplicated().sum())
print("Duplicate Order IDs:", df["Order ID"].nunique(), "unique orders")

## 5. Data Cleaning & Feature Engineering

In [ ]:
from python.data_cleaning import clean_superstore
cleaned = clean_superstore()
cleaned.head()

In [ ]:
cleaned[["sales", "quantity", "discount", "profit", "ship_days", "profit_margin"]].describe().T

## 6. Key Business KPIs

In [ ]:
total_sales = cleaned["sales"].sum()
total_profit = cleaned["profit"].sum()
total_orders = cleaned["order_id"].nunique()
total_quantity = cleaned["quantity"].sum()
profit_margin = total_profit / total_sales
aov = total_sales / total_orders

kpis = pd.Series({
    "Total Sales": total_sales,
    "Total Profit": total_profit,
    "Total Orders": total_orders,
    "Total Quantity": total_quantity,
    "Profit Margin": profit_margin,
    "Average Order Value": aov
})
kpis

## 7. Sales & Profit Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cleaned["sales"].plot(kind="hist", bins=50, ax=ax)
ax.set_title("Distribution of Order-Line Sales")
ax.set_xlabel("Sales")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cleaned["profit"].plot(kind="hist", bins=50, ax=ax)
ax.set_title("Distribution of Order-Line Profit")
ax.set_xlabel("Profit")
plt.tight_layout()
plt.show()

## 8. Regional Performance

In [ ]:
region_perf = cleaned.groupby("region").agg(
    sales=("sales", "sum"), profit=("profit", "sum"),
    orders=("order_id", "nunique"), quantity=("quantity", "sum")
).sort_values("sales", ascending=False)
region_perf["profit_margin"] = region_perf["profit"] / region_perf["sales"]
region_perf

In [ ]:
region_perf[["sales", "profit"]].plot(kind="bar", figsize=(8, 4), title="Sales and Profit by Region")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. State & City Performance

In [ ]:
state_perf = cleaned.groupby("state_province").agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique")).sort_values("profit")
print("Bottom 10 states by profit")
display(state_perf.head(10))
print("\nTop 10 states by sales")
display(state_perf.sort_values("sales", ascending=False).head(10))

In [ ]:
city_perf = cleaned.groupby("city").agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique"))
print("Top 10 cities by sales")
display(city_perf.sort_values("sales", ascending=False).head(10))
print("\nBottom 10 cities by profit")
display(city_perf.sort_values("profit").head(10))

## 10. Category & Sub-Category Performance

In [ ]:
category_perf = cleaned.groupby("category").agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique"), quantity=("quantity","sum"))
category_perf["profit_margin"] = category_perf["profit"] / category_perf["sales"]
category_perf.sort_values("profit", ascending=False)

In [ ]:
subcategory_perf = cleaned.groupby("sub_category").agg(sales=("sales","sum"), profit=("profit","sum"), quantity=("quantity","sum"))
subcategory_perf["profit_margin"] = subcategory_perf["profit"] / subcategory_perf["sales"]
subcategory_perf.sort_values("profit", ascending=False)

## 11. Product Performance & Optimization

In [ ]:
product_perf = cleaned.groupby(["product_id", "product_name"]).agg(sales=("sales","sum"), profit=("profit","sum"), quantity=("quantity","sum"), avg_discount=("discount","mean"), orders=("order_id","nunique")).reset_index()
product_perf["profit_margin"] = product_perf["profit"] / product_perf["sales"]
print("Top 10 products by sales")
display(product_perf.nlargest(10, "sales"))
print("Bottom 10 products by profit")
display(product_perf.nsmallest(10, "profit"))

In [ ]:
loss_making_products = product_perf[product_perf["profit"] < 0]
loss_share = len(loss_making_products) / len(product_perf)
print(f"Loss-making products: {len(loss_making_products):,} / {len(product_perf):,} ({loss_share:.1%})")

## 12. Discount Analysis

In [ ]:
discount_bins = pd.cut(cleaned["discount"], bins=[-0.001,0,0.10,0.20,0.30,0.50,0.80], labels=["0%","1-10%","11-20%","21-30%","31-50%","51-80%"])
discount_perf = cleaned.groupby(discount_bins, observed=False).agg(sales=("sales","sum"), profit=("profit","sum"), quantity=("quantity","sum"))
discount_perf["profit_margin"] = discount_perf["profit"] / discount_perf["sales"]
discount_perf

In [ ]:
discount_corr = cleaned[["discount", "profit", "sales", "quantity"]].corr(numeric_only=True)
discount_corr

## 13. Monthly & Yearly Trends

In [ ]:
monthly = cleaned.groupby("year_month").agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique")).reset_index()
monthly["profit_margin"] = monthly["profit"] / monthly["sales"]
monthly.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(monthly["year_month"], monthly["sales"], label="Sales")
ax.plot(monthly["year_month"], monthly["profit"], label="Profit")
ax.set_title("Monthly Sales and Profit Trend")
ax.tick_params(axis="x", rotation=60)
ax.legend()
plt.tight_layout()
plt.show()

## 14. Segment & Ship Mode Analysis

In [ ]:
segment_perf = cleaned.groupby("segment").agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique"))
segment_perf["profit_margin"] = segment_perf["profit"] / segment_perf["sales"]
segment_perf

In [ ]:
ship_perf = cleaned.groupby("ship_mode").agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique"))
ship_perf["profit_margin"] = ship_perf["profit"] / ship_perf["sales"]
ship_perf

## 15. High-Sales / Low-Profit Opportunities

In [ ]:
sales_threshold = product_perf["sales"].quantile(0.75)
low_margin_threshold = 0.05
opportunities = product_perf[(product_perf["sales"] >= sales_threshold) & (product_perf["profit_margin"] < low_margin_threshold)].sort_values("sales", ascending=False)
print("High-sales / low-margin products:", len(opportunities))
display(opportunities.head(20))

## 16. Business Findings

In [ ]:
west_sales_share = region_perf.loc["West", "sales"] / total_sales
technology_margin = category_perf.loc["Technology", "profit"] / category_perf.loc["Technology", "sales"]
high_discount_rows = (cleaned["discount"] > 0.30).mean()
print(f"West sales share: {west_sales_share:.1%}")
print(f"Technology profit margin: {technology_margin:.1%}")
print(f"Order lines with discount >30%: {high_discount_rows:.1%}")
print(f"Loss-making products: {loss_share:.1%}")

### Key interpretation
- West is the largest sales region, so regional performance should be monitored alongside its profit contribution.
- Technology is the strongest profit contributor among the three categories and maintains a higher margin than the overall business.
- A meaningful subset of products is loss-making, creating a direct product-level optimization opportunity.
- Higher-discount transactions should be examined alongside margin rather than sales alone.
- These findings feed the SQL result tables and the Power BI optimization pages.

## 17. Final Conclusion

The Python analysis establishes the analytical foundation for the project. It validates the dataset, creates reusable business metrics, identifies sales and profitability patterns, and surfaces product, market, and discount opportunities. The resulting insights are presented interactively in the Power BI dashboard, while MySQL provides the structured business-analysis layer.